In [ ]:
# Import the libraries 

import random
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# Use Apple's GPU if available
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.14.1
Device: mps


In [15]:
# Step 3: Load and prepare CIFAR-10

# Image normalization values
mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

# Transformations applied to training images
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Validation and test images are not randomly changed
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Download the dataset
train_dataset_aug = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

train_dataset_eval = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=eval_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=eval_transform
)

# Create a reproducible 90% training / 10% validation split
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

indices = torch.randperm(len(train_dataset_aug)).tolist()
val_size = 5000

val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_subset = Subset(train_dataset_aug, train_indices)
val_subset = Subset(train_dataset_eval, val_indices)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Test images:", len(test_dataset))

100.0%


Training images: 45000
Validation images: 5000
Test images: 10000


In [16]:
# Step 4: Prepare batches of images

BATCH_SIZE = 64

def make_loaders(batch_size=64):
    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders(BATCH_SIZE)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 704
Validation batches: 79
Test batches: 157


In [7]:
class VGG(nn.Module):
    def __init__(self, num_classes=10, activation_layer=nn.ReLU):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            activation_layer(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            activation_layer(),

            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            activation_layer(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            activation_layer(),

            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        return self.classifier(x)

In [18]:
# Check that the model accepts images and returns 10 class scores

images, labels = next(iter(train_loader))
images = images.to(device)

with torch.no_grad():
    outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Device:", device)

Input shape: torch.Size([64, 3, 32, 32])
Output shape: torch.Size([64, 10])
Device: mps


In [19]:
# Step 7: Training and evaluation functions

criterion = nn.CrossEntropyLoss()


def evaluate(model, data_loader):
    model.eval()

    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = 100.0 * correct / total
    average_loss = total_loss / total

    return average_loss, accuracy


def train_model(model, train_loader, val_loader,
                optimizer, epochs=10):

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_accuracy = 0.0
    best_epoch = 0
    best_weights = None

    for epoch in range(1, epochs + 1):
        model.train()

        correct = 0
        total = 0
        total_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Clear previous gradients
            optimizer.zero_grad()

            # Predict the image classes
            outputs = model(images)

            # Calculate how wrong the predictions are
            loss = criterion(outputs, labels)

            # Calculate gradients and update the model
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_accuracy = 100.0 * correct / total

        val_loss, val_accuracy = evaluate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_accuracy)

        # Save the model whenever validation accuracy improves
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_epoch = epoch
            best_weights = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.2f}% | "
            f"Validation Loss: {val_loss:.4f} | "
            f"Validation Accuracy: {val_accuracy:.2f}%"
        )

    if best_weights is not None:
        model.load_state_dict(best_weights)

    print(f"\nBest validation accuracy: {best_val_accuracy:.2f}%")
    print(f"Best epoch: {best_epoch}")

    return history, best_val_accuracy, best_epoch

In [6]:
# Step 8: First experiment — baseline configuration

learning_rate = 0.001
epochs = 10

model = VGG(activation_layer=nn.Sigmoid).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=learning_rate
)

history, best_accuracy, best_epoch = train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    epochs=epochs
)

# Save the best model
torch.save(
    model.state_dict(),
    "vgg6_baseline_best.pth"
)

print("Baseline experiment completed.")

NameError: name 'VGG' is not defined

In [ ]:
# Step 9: Second experiment — different activation functions - 2a
activation_functions = {
    "ReLU": nn.ReLU,
    "Sigmoid": nn.Sigmoid,
    "Tanh": nn.Tanh,
    "SiLU": nn.SiLU,
    "GELU": nn.GELU
}

activation_results = {}

for name, activation in activation_functions.items():

    print(f"\n========== Training with {name} ==========")

    # Reset random seeds for a fair comparison
    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)

    model = VGG(activation_layer=activation).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    history, best_accuracy, best_epoch = train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        epochs=10
    )

    test_loss, test_accuracy = evaluate(model, test_loader)

    activation_results[name] = {
        "history": history,
        "best_val_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "test_accuracy": test_accuracy,
        "test_loss": test_loss
    }

    print(f"{name} Test Accuracy: {test_accuracy:.2f}%")


========== Training with ReLU ==========
Epoch 01/10 | Train Loss: 1.4204 | Train Accuracy: 48.53% | Validation Loss: 1.3386 | Validation Accuracy: 54.24%
Epoch 02/10 | Train Loss: 1.0810 | Train Accuracy: 61.86% | Validation Loss: 1.5679 | Validation Accuracy: 52.42%
Epoch 03/10 | Train Loss: 0.9365 | Train Accuracy: 67.39% | Validation Loss: 0.9787 | Validation Accuracy: 65.50%
Epoch 04/10 | Train Loss: 0.8566 | Train Accuracy: 69.92% | Validation Loss: 0.8748 | Validation Accuracy: 69.76%
Epoch 05/10 | Train Loss: 0.7846 | Train Accuracy: 72.54% | Validation Loss: 0.9127 | Validation Accuracy: 68.68%
Epoch 06/10 | Train Loss: 0.7329 | Train Accuracy: 74.63% | Validation Loss: 1.0688 | Validation Accuracy: 65.66%
Epoch 07/10 | Train Loss: 0.6914 | Train Accuracy: 76.03% | Validation Loss: 0.8288 | Validation Accuracy: 71.46%
Epoch 08/10 | Train Loss: 0.6560 | Train Accuracy: 77.37% | Validation Loss: 0.8920 | Validation Accuracy: 70.36%
Epoch 09/10 | Train Loss: 0.6282 | Train Accur

In [22]:
test_loss, test_accuracy = evaluate(model, test_loader)

print(f"Final Test Loss: {test_loss:.4f}")
print(f"Final Test Top-1 Accuracy: {test_accuracy:.2f}%")

Final Test Loss: 0.6760
Final Test Top-1 Accuracy: 76.83%


In [25]:
optimizer_configs = {
    "SGD": lambda params: optim.SGD(
        params, lr=0.01
    ),

    "Nesterov-SGD": lambda params: optim.SGD(
        params, lr=0.01, momentum=0.9, nesterov=True
    ),

    "Adam": lambda params: optim.Adam(
        params, lr=0.001
    ),

    "Adagrad": lambda params: optim.Adagrad(
        params, lr=0.01
    ),

    "RMSprop": lambda params: optim.RMSprop(
        params, lr=0.001, momentum=0.9
    ),

    "Nadam": lambda params: optim.NAdam(
        params, lr=0.001
    )
}

optimizer_results = {}

for name, optimizer_function in optimizer_configs.items():

    print(f"\n========== Training with {name} ==========")

    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)

    model = VGG(activation_layer=nn.ReLU).to(device)

    optimizer = optimizer_function(model.parameters())

    history, best_accuracy, best_epoch = train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        epochs=10
    )

    test_loss, test_accuracy = evaluate(model, test_loader)

    optimizer_results[name] = {
        "history": history,
        "best_val_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "test_accuracy": test_accuracy,
        "test_loss": test_loss
    }

    print(f"{name} Test Accuracy: {test_accuracy:.2f}%")


========== Training with SGD ==========
Epoch 01/10 | Train Loss: 1.7343 | Train Accuracy: 37.57% | Validation Loss: 1.8581 | Validation Accuracy: 32.42%
Epoch 02/10 | Train Loss: 1.4443 | Train Accuracy: 48.28% | Validation Loss: 1.5507 | Validation Accuracy: 44.92%
Epoch 03/10 | Train Loss: 1.3159 | Train Accuracy: 53.10% | Validation Loss: 1.5571 | Validation Accuracy: 43.02%
Epoch 04/10 | Train Loss: 1.2298 | Train Accuracy: 56.22% | Validation Loss: 1.9870 | Validation Accuracy: 38.22%
Epoch 05/10 | Train Loss: 1.1668 | Train Accuracy: 58.74% | Validation Loss: 1.5199 | Validation Accuracy: 49.92%
Epoch 06/10 | Train Loss: 1.1094 | Train Accuracy: 60.91% | Validation Loss: 2.4155 | Validation Accuracy: 35.44%
Epoch 07/10 | Train Loss: 1.0676 | Train Accuracy: 62.47% | Validation Loss: 2.3165 | Validation Accuracy: 33.78%
Epoch 08/10 | Train Loss: 1.0300 | Train Accuracy: 64.00% | Validation Loss: 2.2297 | Validation Accuracy: 41.44%
Epoch 09/10 | Train Loss: 0.9926 | Train Accura

In [ ]:
# Step 10: Summarize the results of the optimizer experiments
import pandas as pd

optimizer_summary = pd.DataFrame([
    {
        "Optimizer": name,
        "Best Validation Accuracy": round(
            result["best_val_accuracy"], 2
        ),
        "Best Epoch": result["best_epoch"],
        "Test Accuracy": round(
            result["test_accuracy"], 2
        ),
        "Test Loss": round(
            result["test_loss"], 4
        )
    }
    for name, result in optimizer_results.items()
])

display(optimizer_summary)

,Optimizer,Best Validation Accuracy,Best Epoch,Test Accuracy,Test Loss
0,SGD,59.00,10,58.50,1.2097
1,Nesterov-SGD,74.72,10,73.50,0.8147
2,Adam,75.54,9,74.44,0.7647
3,Adagrad,69.64,10,69.34,0.9033
4,RMSprop,77.82,10,76.82,0.7554
5,Nadam,75.18,10,74.47,0.7863


In [31]:
from torch.utils.data import DataLoader

batch_sizes = [32]
batch_results = {}

for batch_size in batch_sizes:

    print(f"\n===== Batch Size: {batch_size} =====")

    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)

    train_loader_exp = DataLoader(
        train_loader.dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader_exp = DataLoader(
        val_loader.dataset,
        batch_size=batch_size,
        shuffle=False
    )

    test_loader_exp = DataLoader(
        test_loader.dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model = VGG(activation_layer=nn.ReLU).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    history, best_accuracy, best_epoch = train_model(
        model,
        train_loader_exp,
        val_loader_exp,
        optimizer,
        epochs=5
    )

    test_loss, test_accuracy = evaluate(
        model, test_loader_exp
    )

    batch_results[batch_size] = {
        "best_val_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "test_accuracy": test_accuracy,
        "test_loss": test_loss,
        "history": history
    }

    print(f"Test Accuracy: {test_accuracy:.2f}%")


===== Batch Size: 32 =====
Epoch 01/5 | Train Loss: 1.4158 | Train Accuracy: 48.53% | Validation Loss: 1.2401 | Validation Accuracy: 56.06%
Epoch 02/5 | Train Loss: 1.0846 | Train Accuracy: 61.72% | Validation Loss: 1.1175 | Validation Accuracy: 62.00%
Epoch 03/5 | Train Loss: 0.9418 | Train Accuracy: 67.03% | Validation Loss: 1.0212 | Validation Accuracy: 64.32%
Epoch 04/5 | Train Loss: 0.8467 | Train Accuracy: 70.52% | Validation Loss: 0.9061 | Validation Accuracy: 68.32%
Epoch 05/5 | Train Loss: 0.7783 | Train Accuracy: 72.81% | Validation Loss: 0.8469 | Validation Accuracy: 70.28%

Best validation accuracy: 70.28%
Best epoch: 5
Test Accuracy: 69.99%


In [32]:
pd.DataFrame([
    {
        "Batch Size": size,
        "Best Validation Accuracy": round(r["best_val_accuracy"], 2),
        "Best Epoch": r["best_epoch"],
        "Test Accuracy": round(r["test_accuracy"], 2),
        "Test Loss": round(r["test_loss"], 4)
    }
    for size, r in batch_results.items()
])

,Batch Size,Best Validation Accuracy,Best Epoch,Test Accuracy,Test Loss
0,32,70.28,5,69.99,0.8681


In [33]:
#change learning rate 
learning_rates = [0.0001, 0.001, 0.01]

lr_results = {}

for lr in learning_rates:

    print(f"\n===== Learning Rate: {lr} =====")

    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)

    model = VGG(activation_layer=nn.ReLU).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr
    )

    history, best_accuracy, best_epoch = train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        epochs=10
    )

    test_loss, test_accuracy = evaluate(model, test_loader)

    lr_results[lr] = {
        "best_val_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "test_accuracy": test_accuracy,
        "test_loss": test_loss,
        "history": history
    }

    print(f"Test Accuracy: {test_accuracy:.2f}%")


===== Learning Rate: 0.0001 =====
Epoch 01/10 | Train Loss: 1.6498 | Train Accuracy: 42.41% | Validation Loss: 1.4336 | Validation Accuracy: 48.70%
Epoch 02/10 | Train Loss: 1.3575 | Train Accuracy: 53.03% | Validation Loss: 1.3161 | Validation Accuracy: 53.14%
Epoch 03/10 | Train Loss: 1.2321 | Train Accuracy: 57.44% | Validation Loss: 1.1717 | Validation Accuracy: 58.64%
Epoch 04/10 | Train Loss: 1.1495 | Train Accuracy: 60.12% | Validation Loss: 1.1154 | Validation Accuracy: 60.36%
Epoch 05/10 | Train Loss: 1.0874 | Train Accuracy: 62.34% | Validation Loss: 1.0930 | Validation Accuracy: 62.10%
Epoch 06/10 | Train Loss: 1.0382 | Train Accuracy: 64.11% | Validation Loss: 1.0673 | Validation Accuracy: 62.32%
Epoch 07/10 | Train Loss: 0.9958 | Train Accuracy: 65.65% | Validation Loss: 1.0349 | Validation Accuracy: 64.06%
Epoch 08/10 | Train Loss: 0.9616 | Train Accuracy: 66.61% | Validation Loss: 0.9916 | Validation Accuracy: 65.00%
Epoch 09/10 | Train Loss: 0.9318 | Train Accuracy: 67

In [34]:
# Step 10: Summarize the results of the learning rate experiments
pd.DataFrame([
    {
        "Learning Rate": lr,
        "Best Validation Accuracy": round(r["best_val_accuracy"], 2),
        "Best Epoch": r["best_epoch"],
        "Test Accuracy": round(r["test_accuracy"], 2),
        "Test Loss": round(r["test_loss"], 4)
    }
    for lr, r in lr_results.items()
])

,Learning Rate,Best Validation Accuracy,Best Epoch,Test Accuracy,Test Loss
0,0.0001,67.34,9,66.73,0.9369
1,0.0010,75.54,9,74.44,0.7647
2,0.0100,79.62,10,79.21,0.6760


In [1]:
import wandb
print("Logged in:", wandb.login())

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/nivedha/.netrc.
wandb: Currently logged in as: cs25m522 (cs25m522-iit-madras) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Logged in: True


In [2]:
sweep_config = {
    "method": "random",

    "metric": {
        "name": "best_val_acc",
        "goal": "maximize"
    },

    "parameters": {
        "activation": {
            "values": ["ReLU", "Tanh", "SiLU", "GELU"]
        },

        "optimizer": {
            "values": ["SGD", "Nesterov-SGD", "Adam",
                       "Adagrad", "RMSprop", "Nadam"]
        },

        "batch_size": {
            "values": [32, 64, 128]
        },

        "epochs": {
            "values": [5, 10, 20]
        },

        "learning_rate": {
            "values": [0.0001, 0.001, 0.01]
        },

        "weight_decay": {
            "values": [0.0, 0.0001]
        }
    }
}

sweep_id = wandb.sweep(
    sweep_config,
    project="VGG6-CIFAR10"
)

print("Sweep ID:", sweep_id)

Create sweep with ID: 10jweaxd
Sweep URL: https://forge.coreweave.com/wandb/cs25m522-iit-madras/VGG6-CIFAR10/sweeps/10jweaxd
Sweep ID: 10jweaxd


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import numpy as np
import random
import pandas as pd

from torch.utils.data import DataLoader

print("Imports loaded successfully!")

Imports loaded successfully!


In [4]:
def train_sweep():
    
    with wandb.init() as run:

        config = wandb.config

        # Activation function
        activation_functions = {
            "ReLU": nn.ReLU,
            "Tanh": nn.Tanh,
            "SiLU": nn.SiLU,
            "GELU": nn.GELU
        }

        model = VGG(
            activation_layer=activation_functions[config.activation]
        ).to(device)

        # DataLoaders
        train_loader_exp = DataLoader(
            train_loader.dataset,
            batch_size=config.batch_size,
            shuffle=True
        )

        val_loader_exp = DataLoader(
            val_loader.dataset,
            batch_size=config.batch_size,
            shuffle=False
        )

        test_loader_exp = DataLoader(
            test_loader.dataset,
            batch_size=config.batch_size,
            shuffle=False
        )

        # Optimizer
        if config.optimizer == "SGD":
            optimizer = optim.SGD(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        elif config.optimizer == "Nesterov-SGD":
            optimizer = optim.SGD(
                model.parameters(),
                lr=config.learning_rate,
                momentum=0.9,
                nesterov=True,
                weight_decay=config.weight_decay
            )

        elif config.optimizer == "Adam":
            optimizer = optim.Adam(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        elif config.optimizer == "Adagrad":
            optimizer = optim.Adagrad(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        elif config.optimizer == "RMSprop":
            optimizer = optim.RMSprop(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        elif config.optimizer == "Nadam":
            optimizer = optim.NAdam(
                model.parameters(),
                lr=config.learning_rate,
                weight_decay=config.weight_decay
            )

        criterion = nn.CrossEntropyLoss()

        best_val_acc = 0
        best_weights = None

        # Training
        for epoch in range(config.epochs):

            model.train()

            running_loss = 0
            correct = 0
            total = 0

            for images, labels in train_loader_exp:

                images = images.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                outputs = model(images)
                loss = criterion(outputs, labels)

                loss.backward()
                optimizer.step()

                running_loss += loss.item() * labels.size(0)

                predictions = outputs.argmax(dim=1)

                correct += (predictions == labels).sum().item()
                total += labels.size(0)

            train_loss = running_loss / total
            train_acc = 100 * correct / total

            # Validation
            model.eval()

            val_loss_total = 0
            val_correct = 0
            val_total = 0

            with torch.no_grad():

                for images, labels in val_loader_exp:

                    images = images.to(device)
                    labels = labels.to(device)

                    outputs = model(images)
                    loss = criterion(outputs, labels)

                    val_loss_total += loss.item() * labels.size(0)

                    predictions = outputs.argmax(dim=1)

                    val_correct += (predictions == labels).sum().item()
                    val_total += labels.size(0)

            val_loss = val_loss_total / val_total
            val_acc = 100 * val_correct / val_total

            if val_acc > best_val_acc:

                best_val_acc = val_acc

                best_weights = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }

            # Log metrics after every epoch
            wandb.log({
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "epoch": epoch + 1
            }, step=epoch + 1)

            print(
                f"Epoch {epoch+1}/{config.epochs} | "
                f"Train Accuracy: {train_acc:.2f}% | "
                f"Validation Accuracy: {val_acc:.2f}%"
            )

        # Restore best model
        if best_weights is not None:
            model.load_state_dict(best_weights)

        test_loss, test_accuracy = evaluate(
            model, test_loader_exp
        )

        wandb.log({
            "best_val_acc": best_val_acc,
            "test_acc": test_accuracy,
            "test_loss": test_loss
        })

        print("Best Validation Accuracy:", best_val_acc)
        print("Test Accuracy:", test_accuracy)

In [8]:
wandb.agent(
    sweep_id,
    function=train_sweep,
    count=5
)

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/nivedha/.netrc.
wandb: Agent Starting Run: 80xwq4we with config:
wandb: 	activation: SiLU
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	learning_rate: 0.0001
wandb: 	optimizer: Nadam
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/nivedha/.netrc.


Traceback (most recent call last):
  File "/var/folders/r8/xfcmty7j4pj8b9wlylm04b5m0000gn/T/ipykernel_97601/2077860352.py", line 17, in train_sweep
    ).to(device)
         ^^^^^^
NameError: name 'device' is not defined


Traceback (most recent call last):
  File "/Users/nivedha/Documents/Personal/IITM/SEM3/DeepLearningLectures/Assignment1/.venv/lib/python3.14/site-packages/wandb/agents/pyagent.py", line 388, in _run_job
    self._function()
    ~~~~~~~~~~~~~~^^
  File "/var/folders/r8/xfcmty7j4pj8b9wlylm04b5m0000gn/T/ipykernel_97601/2077860352.py", line 17, in train_sweep
    ).to(device)
         ^^^^^^
NameError: name 'device' is not defined

wandb: ERROR Run 80xwq4we errored: name 'device' is not defined
wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: l8ohvcz0 with config:
wandb: 	activation: Tanh
wandb: 	batch_size: 128
wandb: 	epochs: 5
wandb: 	learning_rate: 0.001
wandb: 	optimizer: RMSprop
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/nivedha/.netrc.


Traceback (most recent call last):
  File "/var/folders/r8/xfcmty7j4pj8b9wlylm04b5m0000gn/T/ipykernel_97601/2077860352.py", line 17, in train_sweep
    ).to(device)
         ^^^^^^
NameError: name 'device' is not defined


Traceback (most recent call last):
  File "/Users/nivedha/Documents/Personal/IITM/SEM3/DeepLearningLectures/Assignment1/.venv/lib/python3.14/site-packages/wandb/agents/pyagent.py", line 388, in _run_job
    self._function()
    ~~~~~~~~~~~~~~^^
  File "/var/folders/r8/xfcmty7j4pj8b9wlylm04b5m0000gn/T/ipykernel_97601/2077860352.py", line 17, in train_sweep
    ).to(device)
         ^^^^^^
NameError: name 'device' is not defined

wandb: ERROR Run l8ohvcz0 errored: name 'device' is not defined
wandb: Agent Starting Run: i3k0snph with config:
wandb: 	activation: SiLU
wandb: 	batch_size: 64
wandb: 	epochs: 20
wandb: 	learning_rate: 0.0001
wandb: 	optimizer: Adagrad
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /Users/nivedha/.netrc.


Traceback (most recent call last):
  File "/var/folders/r8/xfcmty7j4pj8b9wlylm04b5m0000gn/T/ipykernel_97601/2077860352.py", line 17, in train_sweep
    ).to(device)
         ^^^^^^
NameError: name 'device' is not defined


Traceback (most recent call last):
  File "/Users/nivedha/Documents/Personal/IITM/SEM3/DeepLearningLectures/Assignment1/.venv/lib/python3.14/site-packages/wandb/agents/pyagent.py", line 388, in _run_job
    self._function()
    ~~~~~~~~~~~~~~^^
  File "/var/folders/r8/xfcmty7j4pj8b9wlylm04b5m0000gn/T/ipykernel_97601/2077860352.py", line 17, in train_sweep
    ).to(device)
         ^^^^^^
NameError: name 'device' is not defined

wandb: ERROR Run i3k0snph errored: name 'device' is not defined
wandb: ERROR Detected 3 failed runs in the first 60 seconds, killing sweep.
wandb: To disable this check set WANDB_AGENT_DISABLE_FLAPPING=true
